In [1]:
import os
print(os.getcwd())

/Users/yajatchowdary/Documents/GitHub/careloop-no-show-guard/notebooks


In [2]:
import csv
import json

path = "../data/appointments.csv"

with open(path, newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print("Header:", reader.fieldnames)
print("Row count:", len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020


In [4]:
def parse_row(row):
    parsed = dict(row)  # copy it so that the original rows stays untouched
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed


print(rows[0])
print(parse_row(rows[0]))

{'appointment_id': 'A0001', 'patient_id': 'P509', 'clinic_id': 'C05', 'booked_on': '2025-01-24', 'appt_date': '2025-01-25', 'age': '60', 'gender': 'F', 'distance_km': '18.0', 'fee_inr': '850', 'booking_channel': 'Phone', 'visit_type': 'Consultation', 'reminder_sent': 'Yes', 'prior_no_shows': '1', 'status': 'No-Show'}
{'appointment_id': 'A0001', 'patient_id': 'P509', 'clinic_id': 'C05', 'booked_on': '2025-01-24', 'appt_date': '2025-01-25', 'age': 60, 'gender': 'F', 'distance_km': 18.0, 'fee_inr': 850, 'booking_channel': 'Phone', 'visit_type': 'Consultation', 'reminder_sent': 'Yes', 'prior_no_shows': 1, 'status': 'No-Show'}


In [6]:
accepted = []
rejected = []

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)  # we have to keep the original row as it was

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

# alsp knowing why rows were rejected
blank_age = sum(1 for r in rejected if r["age"].strip() == "")
blank_dist = sum(1 for r in rejected if r["distance_km"].strip() == "")
print("Blank age:", blank_age, "| Blank distance:", blank_dist)

with open("../outputs/rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

Accepted: 988
Rejected: 32
Blank age: 12 | Blank distance: 20


In [8]:
import pandas as pd

df = pd.read_csv("../data/appointments.csv")

print("Shape:", df.shape)
df.info()
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("Raw booking_channel spellings:", df["booking_channel"].nunique())

Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
a

In [9]:
df = df.drop_duplicates()
print(len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(df["booking_channel"].unique())
print(len(df))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print(df["appt_date"].isnull().sum())
print(len(df))

df = df.dropna(subset=["appt_date", "status"])
print(len(df))

1000
<StringArray>
['Phone', 'Walk-In', 'App']
Length: 3, dtype: str
1000
6
1000
990


In [10]:
bad_age = (df["age"] < 1) | (df["age"] > 100)
print(bad_age.sum())
df.loc[bad_age, "age"] = None
print(df["age"].isnull().sum())

print(df["age"].median(), df["distance_km"].median())

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())
print(df[["age", "distance_km"]].isnull().sum())

5
17
48.0 8.25
age            0
distance_km    0
dtype: int64


In [11]:
import numpy as np

for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    capped = ((df[col] < low) | (df[col] > high)).sum()
    print(col, q1, q3, high, "capped:", capped)
    df[col] = np.clip(df[col], low, high)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print(df["fee_inr"].max())

distance_km 4.625 13.0 25.5625 capped: 26
fee_inr 500.0 787.5 1218.75 capped: 3
1219


In [12]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print(df["lead_days"].min(), df["lead_days"].max())
print(df["no_show"].sum(), round(df["no_show"].mean() * 100, 1))

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print(X.mean(axis=0))

mask = X[:, 3] > 14
print(mask.sum(), round(mask.mean() * 100, 1))

x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print((np.abs(z) > 2).sum())

1 45
218 22.0
[ 48.17373737   9.50090909 635.71414141  11.81919192]
296 29.9
53


In [13]:
# now task 1.5, to enrch the clinics, used open meteo here
import requests

clinics = pd.read_csv("../data/clinics.csv")
print(clinics)

def get_temp(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": lat, "longitude": lon, "hourly": "temperature_2m", "forecast_days": 1}
    for attempt in range(2):
        try:
            r = requests.get(url, params=params, timeout=10)
            print(r.status_code)
            if r.status_code == 200:
                data = r.json()
                if "hourly" in data:
                    return data["hourly"]["temperature_2m"][0], None
                reason = "hourly missing"
            else:
                reason = f"status {r.status_code}"
        except requests.exceptions.RequestException as e:
            reason = str(e)
    return None, reason

temps = []
for _, row in clinics.iterrows():
    temp, reason = get_temp(row["latitude"], row["longitude"])
    if temp is None:
        print(row["name"], "failed:", reason)
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("../outputs/clinics_enriched.csv", index=False)
print(clinics)

  clinic_id               clinic_name     city    tier  latitude  longitude
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462
200
200
200
200
200
200
200
200
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur 